# AI model comparison - EDA walkthrough
Run `python -m aimodels all` first (or `--offline` for the synthetic demo). Set `SAMPLE = True` below to use the demo database.
Data: Artificial Analysis (artificialanalysis.ai), OpenRouter, Hugging Face Hub.

In [ ]:
import sys; sys.path.insert(0, "../src")
from pathlib import Path
import pandas as pd
from IPython.display import Image, Markdown, display
from aimodels import eda
from aimodels.config import DB_PATH, SAMPLE_DB_PATH
from aimodels.load import read_last_run

SAMPLE = True            # False -> real data in data/models.db
db = (Path("..") / "data" / SAMPLE_DB_PATH.name) if SAMPLE else (Path("..") / "data" / DB_PATH.name)
out = Path("figures_nb"); out.mkdir(exist_ok=True)
eda._style()
df = eda.load_frame(db)
run = read_last_run(db)
print(run["source"], len(df), "models")
df.head()

## 1. What do we have? Coverage

In [ ]:
cols = ["intelligence_index", "price_blended_usd_per_m", "tokens_per_sec", "params_b", "context_length", "release_date"]
display(df[cols].notna().mean().mul(100).round(1).rename("% present").to_frame())
print("complete on score+price+speed+size:", df[["intelligence_index","price_blended_usd_per_m","tokens_per_sec","params_b"]].notna().all(axis=1).sum())
df["match_method"].value_counts()

## 2. Score vs model size

In [ ]:
path, facts = eda.fig_score_vs_size(df, out, SAMPLE)
print(f"Spearman rho={facts['rho']:.2f} (n={facts['n']}), +{facts['slope_per_10x']:.1f} points per 10x parameters, R2={facts['r2']:.2f}")
Image(path)

## 3. Score vs price

In [ ]:
path, facts = eda.fig_score_vs_price(df, out, SAMPLE)
display(Image(path))
facts["frontier"][["name", "intelligence_index", "price_blended_usd_per_m", "tokens_per_sec"]]

## 4. Speed vs price

In [ ]:
path, facts = eda.fig_speed_vs_price(df, out, SAMPLE)
print(f"Spearman rho (log price vs speed) = {facts['rho']:.2f}")
Image(path)

## 5. Benchmark correlations

In [ ]:
path, facts = eda.fig_correlations(df, out, SAMPLE)
display(Image(path))
pd.DataFrame(facts["pairs"][:5], columns=["A", "B", "rho", "shared n"])

## 6. Trends over time

In [ ]:
path, facts = eda.fig_trends(df, out, SAMPLE)
Image(path)

## 7. Missing data

In [ ]:
path, facts = eda.fig_missing(df, out, SAMPLE)
display(Image(path))
df.groupby("group")[["price_blended_usd_per_m", "tokens_per_sec", "context_length", "params_b", "hle"]].agg(lambda s: s.isna().mean().round(3))

## 8. Best value

In [ ]:
path, facts = eda.fig_value(df, out, SAMPLE)
Image(path)

## Takeaways
*Fill in after running on live data.* Suggested questions: Which models sit on the price and size frontiers? How big is the open-vs-closed gap and is it shrinking? Which benchmarks are redundant with the Intelligence Index? Is missing data random or structural?